In [1]:
import pandas as pd
import numpy as np

In [ ]:
sv_file = pd.read_hdf('../../../SV_sample_data/data/sv_file_dup_cleaned.h5')
included_samples = pd.read_csv("../../../SV_sample_data/data/final_samples.csv")
sv_file = sv_file[sv_file['name'].isin(included_samples['tumor_normal_pair'])]

print("Number of structural variant calls:", len(sv_file))
print("Number of participants:", len(sv_file['name'].unique()))
sv_file

Number of structural variant calls: 65372
Number of participants: 234


,num,chr1,str1,pos1,chr2,str2,pos2,class,span,somatic,somatic_score,SvABA,Manta,VCF_TALT,VCF_NALT,VCF_TREF,VCF_NREF,name
0,614,1,0,3680543,1,1,3731731,deletion,51189.0,1,15.0,0,1,11,0,149,86,09T02-09N01
1,615,1,0,25148899,1,1,25252735,deletion,103843.0,1,15.0,0,1,12,0,126,72,09T02-09N01
2,909,1,0,124154195,1,1,124163217,deletion,9022.0,1,5.0,0,1,9,0,199,69,09T02-09N01
3,719,1,1,160827717,1,0,160890810,tandem_dup,63090.0,1,11.0,0,1,15,0,163,65,09T02-09N01
4,33,1,1,246260290,19,1,6750721,inter_chr,NaN,1,41.0,1,1,25,0,127,73,09T02-09N01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108262,61,20,0,3436041,20,1,3541712,deletion,105671.0,1,16.0,0,1,11,0,186,126,WGS_15_2-WGS_16_2
108263,15,20,1,29963715,20,0,29969207,tandem_dup,5492.0,1,12.0,1,1,10,0,99,51,WGS_15_2-WGS_16_2
108264,1,20,1,46626966,20,1,46643763,inversion,16800.0,1,27.0,1,1,18,0,209,108,WGS_15_2-WGS_16_2
108265,64,20,1,46640237,20,0,46641104,tandem_dup,865.0,1,13.0,0,1,13,1,222,107,WGS_15_2-WGS_16_2


In [3]:
def classify_sv_type_by_strands(chrom1, chrom2, strand1, strand2, original_class=None):
    
    # For translocations (different chromosomes), always use TRA
    if chrom1 != chrom2:
        return 'translocation'
        
    # For same chromosome, encode based on strand orientations
    if strand1 == 0 and strand2 == 1:
        return 'deletion'  # deletion-like (+/-)
    elif strand1 == 1 and strand2 == 0:
        return 'tandem-duplication'  # duplication-like (-/+)
    elif strand1 == 0 and strand2 == 0:
        return 'inversion'  # head-to-head inversion (+/+)
    elif strand1 == 1 and strand2 == 1:
        return 'inversion'  # tail-to-tail inversion (-/-)
    else:
        print(f"Warning: Unusual strand pattern {strand1}/{strand2} for {original_class}")
        return original_class.upper()

# Store the original class for comparison
sv_file['original_class'] = sv_file['class'].copy()

# Apply the function to create new class column
sv_file['new_class'] = sv_file.apply(lambda row: classify_sv_type_by_strands(
    row['chr1'], row['chr2'], row['str1'], row['str2'], row['original_class']
), axis=1)

# Create a cross-tabulation summary
print("Summary of changes (Original -> New):")
summary = pd.crosstab(sv_file['original_class'], sv_file['new_class'], margins=True)
print(summary)
print()

# Replace the original class column with the new classification
sv_file['class'] = sv_file['new_class']
df = sv_file.drop(['original_class', 'new_class'], axis=1)

Summary of changes (Original -> New):
new_class       deletion  inversion  tandem-duplication  translocation    All
original_class                                                               
deletion            6264          0                   0              0   6264
inter_chr              0          0                   0          20537  20537
inversion              0      10194                   0              0  10194
long_range          6032      11446                5663              0  23141
tandem_dup             0          0                5236              0   5236
All                12296      21640               10899          20537  65372



In [4]:
# Convert df to BEDPE format
bedpe_df = pd.DataFrame()

# Required BEDPE columns
bedpe_df['chrom1'] = df['chr1'].astype(str)
bedpe_df['start1'] = df['pos1'] - 1  # Convert to 0-based coordinates
bedpe_df['end1'] = df['pos1']        # End is start + 1 for breakpoint
bedpe_df['chrom2'] = df['chr2'].astype(str)
bedpe_df['start2'] = df['pos2'] - 1  # Convert to 0-based coordinates
bedpe_df['end2'] = df['pos2']        # End is start + 1 for breakpoint
bedpe_df['sample'] = df['name']
bedpe_df['svclass'] = df['class']

print(bedpe_df.head())

# save file
bedpe_df.to_csv('../data/sv_signature_analysis.bedpe', sep='\t', index=False)


  chrom1     start1       end1 chrom2     start2       end2       sample  \
0      1    3680542    3680543      1    3731730    3731731  09T02-09N01   
1      1   25148898   25148899      1   25252734   25252735  09T02-09N01   
2      1  124154194  124154195      1  124163216  124163217  09T02-09N01   
3      1  160827716  160827717      1  160890809  160890810  09T02-09N01   
4      1  246260289  246260290     19    6750720    6750721  09T02-09N01   

              svclass  
0            deletion  
1            deletion  
2            deletion  
3  tandem-duplication  
4       translocation  
